# Chapter 13 — Dimensionality Reduction: PCA & Fisher Discriminant Analysis

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/winnertechserv/aimlgybs/blob/main/part2_classic_ml/ch13_dimensionality_reduction/ch13_notebook.ipynb)

Companion notebook for **Chapter 13** of *AI-Machine Learning: Get Your Base Strong*. Read the chapter first, then run each cell and compare your output with the output printed in the book (shown below each cell).


In [ ]:
# Optional: install the exact library versions used for the outputs in the book.
# Google Colab already has these libraries; newer versions may change a few digits.
# !pip install -q numpy==2.5.3 pandas==3.0.5 scipy==1.18.1 scikit-learn==1.9.1 matplotlib==3.11.2 pillow==12.3.0

## Key takeaways

- PCA finds new axes (principal components) ordered by how much variance they capture, and you keep enough to reach about 95%.
- Eigenvalues tell you the variance along each component; eigenvectors give the directions themselves.
- PCA is unsupervised: it never looks at the class labels, so its best direction may be useless for classification.
- FDA (LDA) is supervised and maximizes class separation, which is why its 1-D projection usually classifies better than PCA's.
- Dimensionality reduction is about removing redundancy, not about throwing away data at random.

## Example 1: PCA Step-by-Step from Scratch

This example builds PCA by hand, step by step, exactly as described in the overview. We have 20 days of five weather readings, and one hidden weather pattern drives all of them, so they are strongly correlated. Because the readings are in very different units (Rain runs into hundreds of mm), we first standardize them, as Chapter 6 taught. Then we compute the covariance matrix, find its eigenvalues and eigenvectors, sort them and apply the 95% rule. PC1 alone captures about 97% of the variance, and its weights show all five readings contributing almost equally: the five features really do carry one shared signal. Finally we project the data onto PC1 (aₙ = Qᵀxₙ) and check that the variance of the projected scores equals the PC1 eigenvalue.

In [1]:
import numpy as np

np.random.seed(0)
N = 20
# One hidden 'weather' factor drives all five readings, plus a little noise.
# So the five features are strongly correlated - that is what PCA can exploit.
z = np.random.normal(0, 1, N)
data = np.column_stack([
    23.5 + 1.5 * z + np.random.normal(0, 0.25, N),   # Temperature (C)
    90.0 - 4.0 * z + np.random.normal(0, 0.70, N),   # Humidity (%)
    1012 - 6.0 * z + np.random.normal(0, 1.00, N),   # Pressure (hPa)
    600 - 400 * z + np.random.normal(0, 60.0, N),    # Rain (mm)
    50.0 - 15 * z + np.random.normal(0, 2.50, N),    # Moisture (%)
])
names = ["Temperature", "Humidity", "Pressure", "Rain", "Moisture"]
print(f"Dataset shape: {data.shape}  ({N} samples, 5 features)\n")

# Step 0: Standardize (Chapter 6). Rain is in hundreds of mm and Temperature in
# tens of degrees; without this step Rain's big units alone would decide PC1.
X_s = (data - data.mean(axis=0)) / data.std(axis=0, ddof=1)
print("Step 0: Features standardized (mean 0, std 1)")

# Step 1: Covariance matrix of the standardized data (= correlation matrix)
cov = np.cov(X_s.T)                      # np.cov divides by N - 1
print("Step 1: Covariance matrix computed (5x5)")
print("        Correlation of Temperature with the others:", np.round(cov[0, 1:], 2))

# Step 2: Eigen decomposition (sorted descending)
evals, evecs = np.linalg.eigh(cov)
idx    = np.argsort(evals)[::-1]
evals  = evals[idx]
evecs  = evecs[:, idx]

print("\nStep 2: Eigenvalues and cumulative variance:")
total = evals.sum()
cum   = 0
for i, ev in enumerate(evals):
    cum += ev
    flag = " <- 95% crossed" if cum/total >= 0.95 and (cum-ev)/total < 0.95 else ""
    print(f"  PC{i+1}: {ev:>7.4f}  |  {ev/total*100:5.2f}%  |  cum {cum/total*100:6.2f}%{flag}")

# Step 3: Choose l with 95% criterion
l = next(i+1 for i in range(len(evals)) if evals[:i+1].sum()/total >= 0.95)
print(f"\nStep 3: Keep l = {l} principal component(s) (95% rule)")
pc1 = evecs[:, 0] * np.sign(evecs[0, 0])   # fix the sign so Temperature is positive
print("        PC1 weights:", {n: round(float(v), 2) for n, v in zip(names, pc1)})

# Step 4: Project data: a_n = Q^T x_n
Q = evecs[:, :l]
projected = X_s @ Q
print(f"Step 4: Original shape {data.shape} -> Projected shape {projected.shape}")
print(f"\nFirst 5 projected samples:")
for i in range(5):
    print(f"  Sample {i+1}: {projected[i, 0]:+.2f}")

# Step 5: the variance along PC1 equals its eigenvalue (both divide by N - 1)
print(f"\nStep 5: Variance of PC1 scores: {projected.var(ddof=1):.4f}"
      f"  (eigenvalue of PC1: {evals[0]:.4f})")

Dataset shape: (20, 5)  (20 samples, 5 features)

Step 0: Features standardized (mean 0, std 1)
Step 1: Covariance matrix computed (5x5)
        Correlation of Temperature with the others: [-0.96 -0.96 -0.96 -0.97]

Step 2: Eigenvalues and cumulative variance:
  PC1:  4.8652  |  97.30%  |  cum  97.30% <- 95% crossed
  PC2:  0.0598  |   1.20%  |  cum  98.50%
  PC3:  0.0311  |   0.62%  |  cum  99.12%
  PC4:  0.0266  |   0.53%  |  cum  99.65%
  PC5:  0.0174  |   0.35%  |  cum 100.00%

Step 3: Keep l = 1 principal component(s) (95% rule)
        PC1 weights: {'Temperature': 0.45, 'Humidity': -0.45, 'Pressure': -0.45, 'Rain': -0.45, 'Moisture': -0.45}
Step 4: Original shape (20, 5) -> Projected shape (20, 1)

First 5 projected samples:
  Sample 1: -2.83
  Sample 2: +0.16
  Sample 3: -1.37
  Sample 4: -4.06
  Sample 5: -3.41

Step 5: Variance of PC1 scores: 4.8652  (eigenvalue of PC1: 4.8652)


## Example 2: PCA with scikit-learn on Handwritten Digits

This is PCA on real data: scikit-learn's handwritten digits, where each image is 8×8 = 64 pixels (the same idea scales to 28×28 MNIST images). We use scikit-learn's PCA to find how many components are needed to keep 95% and 99% of the variance. All pixels share one unit (brightness from 0 to 16), so we do not standardize them. Just 29 of the 64 dimensions keep 95% of the variance, matching Figure 13.2, and the images shrink by more than half. The reconstruction error is small next to the average pixel variance, so the 35 dropped directions carried little information. This is the eigenface idea in action: store a few component scores per image instead of every pixel.

In [2]:
from sklearn.decomposition import PCA
from sklearn.datasets import load_digits
import numpy as np

digits = load_digits()
X = digits.data   # shape: (1797, 64), pixel brightness 0-16

print(f"Original dataset shape : {X.shape}")
print(f"  -> 1797 images, each 8x8 = 64 pixel features\n")

# All pixels share one unit (brightness 0-16), so we do NOT standardize here.
# Standardizing would blow up the near-constant border pixels to the same weight
# as the useful ones. (Note: we fit on all images because this is unsupervised
# compression; before a classifier, fit PCA on the training images only, as in Example 3.)
pca_full = PCA()
pca_full.fit(X)
cumvar = np.cumsum(pca_full.explained_variance_ratio_) * 100

l_95 = np.argmax(cumvar >= 95) + 1
l_99 = np.argmax(cumvar >= 99) + 1

print("Variance explained by top N components:")
for i in [0, 1, 2, 4, 9, 19, l_95-1, l_99-1]:
    print(f"  Top {i+1:>3} PCs -> cumulative variance: {cumvar[i]:.2f}%")

print(f"\n95% threshold: keep {l_95} components")
print(f"99% threshold: keep {l_99} components")
print(f"Reduction: 64 -> {l_95} dims  ({(1 - l_95/64)*100:.1f}% smaller)")

pca = PCA(n_components=l_95)
X_reduced = pca.fit_transform(X)
X_recon   = pca.inverse_transform(X_reduced)
mse = np.mean((X - X_recon) ** 2)

print(f"\nReduced dataset shape  : {X_reduced.shape}")
print(f"Reconstruction MSE     : {mse:.4f}  (average pixel variance: {X.var(axis=0).mean():.2f})")
print("(Small MSE = minimal information lost)")

Original dataset shape : (1797, 64)
  -> 1797 images, each 8x8 = 64 pixel features

Variance explained by top N components:
  Top   1 PCs -> cumulative variance: 14.89%
  Top   2 PCs -> cumulative variance: 28.51%
  Top   3 PCs -> cumulative variance: 40.30%
  Top   5 PCs -> cumulative variance: 54.50%
  Top  10 PCs -> cumulative variance: 73.82%
  Top  20 PCs -> cumulative variance: 89.43%
  Top  29 PCs -> cumulative variance: 95.48%
  Top  41 PCs -> cumulative variance: 99.01%

95% threshold: keep 29 components
99% threshold: keep 41 components
Reduction: 64 -> 29 dims  (54.7% smaller)

Reduced dataset shape  : (1797, 29)
Reconstruction MSE     : 0.8486  (average pixel variance: 18.77)
(Small MSE = minimal information lost)


## Example 3: Fisher Discriminant Analysis (FDA/LDA) vs PCA

This is the key example that proves WHY FDA beats PCA for classification tasks. We use sklearn's LinearDiscriminantAnalysis (which builds FDA) and compare it to PCA on the same 10-feature, 2-class dataset. Both reduce to 1 dimension. FDA finds a direction that maximizes class separation (the class means are 1.96 apart after projection). PCA finds the direction of maximum variance — but that direction might not be useful for distinguishing classes. The accuracy comparison confirms it: FDA 1D is nearly as good as the full 10D dataset, while PCA 1D loses large accuracy.

In [3]:
import numpy as np
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.decomposition import PCA
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

np.random.seed(42)
X, y = make_classification(n_samples=300, n_features=10, n_informative=5,
                            n_redundant=3, n_classes=2, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

print("=== FDA vs PCA for Classification ===\n")
print(f"Dataset: {X.shape[0]} samples, {X.shape[1]} features, 2 classes\n")

# FDA: project to 1D (max class separability)
lda = LinearDiscriminantAnalysis(n_components=1)
X_tr_lda = lda.fit_transform(X_train, y_train)
X_te_lda = lda.transform(X_test)

print("Step 1: FDA projection direction maximizes class separation")
print(f"  Class +1 mean in FDA space: {X_tr_lda[y_train==1].mean():.4f}")
print(f"  Class  0 mean in FDA space: {X_tr_lda[y_train==0].mean():.4f}")
print(f"  Separation between means  : {abs(X_tr_lda[y_train==1].mean() - X_tr_lda[y_train==0].mean()):.4f}")

# PCA: project to 1D (max variance)
pca = PCA(n_components=1)
X_tr_pca = pca.fit_transform(X_train)
X_te_pca = pca.transform(X_test)

def knn_acc(X_tr, X_te, y_tr, y_te):
    clf = KNeighborsClassifier(n_neighbors=5)
    clf.fit(X_tr, y_tr)
    return accuracy_score(y_te, clf.predict(X_te))

print("\nStep 2: Classification accuracy comparison (5-NN classifier)")
print(f"  Original 10D data      : {knn_acc(X_train, X_test, y_train, y_test)*100:.1f}%")
print(f"  PCA 1D projection      : {knn_acc(X_tr_pca, X_te_pca, y_train, y_test)*100:.1f}%  (max variance, may miss class info)")
print(f"  FDA 1D projection      : {knn_acc(X_tr_lda, X_te_lda, y_train, y_test)*100:.1f}%  (max separability)")

print("\nFDA projection weights per feature:")
for i, w in enumerate(lda.coef_[0]):
    print(f"  Feature {i+1:>2}: {w:+.4f}")

=== FDA vs PCA for Classification ===

Dataset: 300 samples, 10 features, 2 classes

Step 1: FDA projection direction maximizes class separation
  Class +1 mean in FDA space: 1.0191
  Class  0 mean in FDA space: -0.9443
  Separation between means  : 1.9633

Step 2: Classification accuracy comparison (5-NN classifier)
  Original 10D data      : 82.2%
  PCA 1D projection      : 73.3%  (max variance, may miss class info)
  FDA 1D projection      : 80.0%  (max separability)

FDA projection weights per feature:
  Feature  1: +0.0033
  Feature  2: -0.1774
  Feature  3: +0.7178
  Feature  4: -0.3141
  Feature  5: +0.6349
  Feature  6: -0.3084
  Feature  7: +0.1813
  Feature  8: -0.4975
  Feature  9: -1.0280
  Feature 10: -0.2835
